# 02 — 推薦モデル動作確認・推薦結果分析

学習・推薦・評価は `src/music_recsys/evaluation/benchmark.py` の `run_experiment` に任せ、
ここでは推薦内容・履歴除外・候補数・人気偏りを確認します。実装済みモデルはPopularityのみです。

合成データで独立実行でき、DBと常駐MLflowは不要です。指標の集計と比較は `03_analysis.ipynb` で扱います。
Music4Allの100曲・100ユーザーsnapshotはタグ探索にのみ使えます。再生関係がないため推薦学習には使いません。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from music_recsys.analysis import popularity_rows, recommendation_rows, request_rows, tag_rows
from music_recsys.data import load_dataset
from music_recsys.evaluation.benchmark import run_experiment
from music_recsys.evaluation.tracking import TrackingConfig

ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "configs/toy.toml").is_file()
)

# 学習・要求実行・評価ロジックは共通Runner。環境変数で有効でも記録を明示的にオフにする。
dataset = load_dataset(ROOT / "examples/toy/dataset.json", top_k=2)
report = run_experiment(ROOT / "configs/toy.toml", tracking=TrackingConfig(enabled=False))
requests = request_rows(report)
recommendations = recommendation_rows(dataset, report)
display(
    {
        "model": report.metadata.model_name,
        "dataset": report.metadata.dataset_id,
        "top_k": report.metadata.top_k,
        "training_seconds": report.metadata.training_seconds,
    }
)

## 要求と推薦結果

要求は `user_id`・`history`・`candidate_ids`・`top_k` を持ちます。正解は別の型で評価器だけに渡します。
表示には要求の連番を使い、ユーザーID・生履歴は含めません。
`is_relevant` は推薦後の分析用で、モデルが正解を参照したことを意味しません。
正解の評価対象がない要求では `is_relevant=None`（未評価）です。Falseと区別します。

In [ ]:
display(requests)
display(recommendations)
assert all(row["history_overlap_count"] == 0 for row in requests)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
indices = [row["request_index"] for row in requests]
axes[0].bar(
    indices, [row["candidate_count"] for row in requests], color="#256b78", label="Candidates"
)
axes[0].bar(indices, [row["returned_count"] for row in requests], color="#c77936", label="Returned")
axes[0].set(title="Candidates and returned tracks", xlabel="Request index", ylabel="Tracks")
axes[0].legend()
axes[1].bar(indices, [row["history_overlap_count"] for row in requests], color="#8b507b")
axes[1].set(
    title="History overlap (expected 0)", xlabel="Request index", ylabel="Tracks", ylim=(0, 1)
)
fig.tight_layout()
plt.show()

## 人気曲への偏り

Popularityは学習中の再生回数で順位を決め、同数の場合は曲IDで決定的に並べます。
学習頻度と推薦への出現回数を並べます。曲IDは合成データのものです。
人気0の曲も候補には入り、履歴除外やtop-kによって結果が変わります。

In [ ]:
popularity = popularity_rows(dataset, report)
items = [row["item_id"] for row in popularity]
fig, axes = plt.subplots(1, 2, figsize=(11, 3), sharex=True)
axes[0].bar(items, [row["train_events"] for row in popularity], color="#256b78")
axes[0].set(title="Training popularity", ylabel="Train events")
axes[1].bar(items, [row["recommended_count"] for row in popularity], color="#c77936")
axes[1].set(title="Recommendation exposure", ylabel="Returned occurrences")
fig.tight_layout()
plt.show()

## 任意：Music4All snapshotのタグ探索

`LOAD_SNAPSHOT=True` と公開済み `SNAPSHOT_ID` の設定で、`load_snapshot` を通じて読込みます。
`01_data.ipynb` の実行済み変数には依存しません。接続設定は環境変数で渡します。
`tag_rows` はタグを持つ曲数と重みの平均を集計し、ユーザーIDを表示しません。
この図はメタデータの探索です。合成推薦結果とMusic4Allのタグを結び付けません。
タグ未取得・空タグの曲にはタグを補完しません。

In [ ]:
LOAD_SNAPSHOT = False
SNAPSHOT_ID = None

if LOAD_SNAPSHOT:
    from uuid import UUID

    from music_recsys.ingestion.pipeline import load_snapshot
    from music_recsys.ingestion.postgres import engine_from_environment

    if SNAPSHOT_ID is None:
        raise ValueError("Set SNAPSHOT_ID before loading a published snapshot")
    engine = engine_from_environment()
    try:
        catalog = load_snapshot(engine, ROOT / "data/imports", UUID(SNAPSHOT_ID))
    finally:
        engine.dispose()
    tags = tag_rows(catalog)
    display({"tracks": len(catalog.tracks), "tag_kinds": len(tags), **catalog.stats})
    display(tags[:20])
    if tags:
        top_tags = tags[:15]
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.barh(
            [row["tag"] for row in top_tags], [row["n_tracks"] for row in top_tags], color="#256b78"
        )
        ax.invert_yaxis()
        ax.set(title="Music4All snapshot: tag coverage", xlabel="Tracks with tag")
        fig.tight_layout()
        plt.show()
else:
    print("Snapshot tag analysis is disabled; synthetic tracks have no supplied tags.")

## 確認事項

- 履歴との重複数、候補外・重複推薦・失敗は `request_rows` の列で確認する。
- 候補よりKが大きい場合の短い出力と、正解を持たない要求は指標分析で扱う。
- 現在の結果は6曲の合成データによる動作確認。実データでの精度・タグ嗜好の妥当性は未評価。